In [6]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# --- CONFIGURACIÓN ---
DATA_PATH = 'Data/full_data_flightdelay.csv' 
ARTIFACTS_DIR = 'artifacts'

def train_and_export():
    print("1. Cargando datos...")
    
    # Verificación de seguridad para la carpeta Data
    if not os.path.exists(DATA_PATH):
        print(f"❌ ERROR: No encuentro el archivo: {DATA_PATH}")
        print("Asegúrate de estar en la carpeta correcta.")
        return

    cols = [
        'MONTH', 'DAY_OF_WEEK', 'DEP_DEL15', 'DEP_TIME_BLK', 'DISTANCE_GROUP', 
        'SEGMENT_NUMBER', 'CONCURRENT_FLIGHTS', 'CARRIER_NAME', 'DEPARTING_AIRPORT', 
        'PRCP', 'TMAX', 'AWND', 'AIRPORT_FLIGHTS_MONTH', 'PLANE_AGE'
    ]
    
    df = pd.read_csv(DATA_PATH, usecols=cols).dropna(subset=['DEP_DEL15']).fillna(0)

    print("2. Procesando y guardando mapas de riesgo...")
    
    # Aseguramos que la carpeta de artefactos exista
    if not os.path.exists(ARTIFACTS_DIR):
        os.makedirs(ARTIFACTS_DIR)
        print(f"   -> Carpeta '{ARTIFACTS_DIR}' creada.")
    
    for col in ['CARRIER_NAME', 'DEPARTING_AIRPORT', 'DEP_TIME_BLK']:
        risk_map = df.groupby(col)['DEP_DEL15'].mean().to_dict()
        df[col + '_RISK'] = df[col].map(risk_map)
        
        # Usamos os.path.join para evitar errores de rutas en Windows/Mac
        save_path = os.path.join(ARTIFACTS_DIR, f'{col}_risk_map.joblib')
        joblib.dump(risk_map, save_path)

    # Definimos las columnas finales
    features_list = [
        'MONTH', 'DAY_OF_WEEK', 'DISTANCE_GROUP', 'SEGMENT_NUMBER', 
        'CONCURRENT_FLIGHTS', 'PRCP', 'TMAX', 'AWND', 'PLANE_AGE',
        'AIRPORT_FLIGHTS_MONTH', 'CARRIER_NAME_RISK', 
        'DEPARTING_AIRPORT_RISK', 'DEP_TIME_BLK_RISK'
    ]

    X = df[features_list]
    y = df['DEP_DEL15']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("3. Entrenando Random Forest...")
    model = RandomForestClassifier(
        n_estimators=60, 
        max_depth=12,
        min_samples_leaf=20,
        max_samples=0.2,
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Validación rápida
    print(f"--- RESULTADOS FINALES ---")
    print(f"Precisión General (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))

    print("4. Exportando modelo a ONNX...")
    initial_type = [('float_input', FloatTensorType([None, len(features_list)]))]
    onnx_model = convert_sklearn(model, initial_types=initial_type)

    onnx_path = os.path.join(ARTIFACTS_DIR, "flight_delay_rf.onnx")
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    print(f"✅ ¡Listo! Modelo guardado en: {onnx_path}")

if __name__ == "__main__":
    train_and_export()

1. Cargando datos...
2. Procesando y guardando mapas de riesgo...
   -> Carpeta 'artifacts' creada.
3. Entrenando Random Forest...
--- RESULTADOS FINALES ---
Precisión General (Accuracy): 0.8170
              precision    recall  f1-score   support

           0       0.82      1.00      0.90   1052339
           1       0.73      0.05      0.09    245474

    accuracy                           0.82   1297813
   macro avg       0.78      0.52      0.50   1297813
weighted avg       0.80      0.82      0.75   1297813

4. Exportando modelo a ONNX...
✅ ¡Listo! Modelo guardado en: artifacts\flight_delay_rf.onnx
